# Tutorial 3: Real Data Analysis

This notebook demonstrates the full real-data pipeline using the demo dataset:
- 20 healthy controls + 15 HNC patients (mild/moderate/severe post-RT)
- Paired clinical scores: FOIS, MBSImP, PAS

Steps:
1. Load and preprocess landmark data
2. Manifold embedding
3. Compute trajectory metrics
4. Correlate with clinical scores
5. Group comparisons with statistics

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.stats import spearmanr, mannwhitneyu

from core.trajectory import SwallowingTrajectory
from core.metrics import extract_all_metrics
from core.phase_detection import detect_phases_geometric, bottleneck_traversal_score
from simulation.trajectory_generator import DEFAULT_LANDMARK_GROUPS

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
groups = DEFAULT_LANDMARK_GROUPS

## 1. Load Data

In [ ]:
df_raw = pd.read_csv('../data/real/raw/demo_swallowing_landmarks.csv')
df_clin = pd.read_csv('../data/real/clinical_scores.csv')

print(f'Landmark data: {len(df_raw)} rows')
print(f'Subjects: {df_raw["subject_id"].nunique()}')
print(f'Conditions: {df_raw["condition"].value_counts().to_dict()}')
print(f'\nClinical scores:')
display(df_clin.head(10))

## 2. Preprocess and Build Trajectories

In [ ]:
coord_cols = [c for c in df_raw.columns if c not in ('subject_id','condition','frame','time')]

trajectories = []
for sid, sdf in df_raw.groupby('subject_id'):
    sdf = sdf.sort_values('frame')
    lm = sdf[coord_cols].values.astype(np.float64)
    t = sdf['time'].values
    cond = sdf['condition'].iloc[0]
    traj = SwallowingTrajectory(lm, t, 25.0, sid, cond).smooth().interpolate(200)
    trajectories.append(traj)

print(f'Loaded {len(trajectories)} trajectories')
print(f'Example: {trajectories[0].n_frames} frames × {trajectories[0].n_dims} dims')

## 3. Manifold Embedding

In [ ]:
X = np.vstack([t.landmarks for t in trajectories])
pca3 = PCA(n_components=3).fit(X)

cmap = {'healthy':'#2ecc71','post_rt_mild':'#f39c12','post_rt_moderate':'#e67e22','post_rt_severe':'#e74c3c'}

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
for traj in trajectories:
    emb = pca3.transform(traj.landmarks)
    ax.plot(emb[:,0], emb[:,1], emb[:,2], color=cmap.get(traj.condition,'gray'), alpha=0.4, lw=0.8)
for c, col in cmap.items():
    ax.plot([],[],[], color=col, lw=2.5, label=c.replace('_',' ').title())
ax.legend(); ax.set_title('Real Data: Trajectory Embedding'); ax.view_init(25, 45)
plt.show()

## 4. Compute Metrics and Merge with Clinical Scores

In [ ]:
metric_rows = []
for traj in trajectories:
    m = extract_all_metrics(traj, groups)
    ph = detect_phases_geometric(traj)
    m['bottleneck_score'] = bottleneck_traversal_score(traj, ph)
    m['subject_id'] = traj.subject_id
    m['condition'] = traj.condition
    metric_rows.append(m)

df_m = pd.DataFrame(metric_rows).merge(df_clin, on=['subject_id','condition'], how='left')
print(f'{len(df_m)} subjects with metrics + clinical scores')
display(df_m[['subject_id','condition','geodesic_length','mean_curvature','bottleneck_score','FOIS','PAS']].head(10))

## 5. Clinical Correlations

In [ ]:
m_cols = ['geodesic_length','mean_curvature','bottleneck_score','smoothness_index']
m_cols = [c for c in m_cols if c in df_m.columns]
c_cols = ['FOIS','MBSImP_oral','MBSImP_pharyngeal','PAS']

print(f'{"Metric":<25s} {"Clinical":>10s} {"ρ":>8s} {"p":>10s}')
print('-' * 55)
for mc in m_cols:
    for cc in c_cols:
        valid = df_m[[mc, cc]].dropna()
        if len(valid) > 5:
            rho, pval = spearmanr(valid[mc], valid[cc])
            sig = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else ''
            if abs(rho) > 0.4:
                print(f'{mc:<25s} {cc:>10s} {rho:>+8.3f} {pval:>10.4f} {sig}')

In [ ]:
# Scatter plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
pairs = [('geodesic_length','FOIS'), ('bottleneck_score','PAS'), ('smoothness_index','MBSImP_pharyngeal')]
for ax, (mc, cc) in zip(axes, pairs):
    for cond in df_m['condition'].unique():
        sub = df_m[df_m['condition']==cond]
        ax.scatter(sub[mc], sub[cc], color=cmap.get(cond,'gray'), s=60, alpha=0.7,
                   label=cond.replace('_',' ').title(), edgecolors='white', lw=0.5)
    rho, p = spearmanr(df_m[mc].dropna(), df_m[cc].dropna())
    ax.set_xlabel(mc.replace('_',' ').title()); ax.set_ylabel(cc)
    ax.set_title(f'ρ = {rho:.3f}, p = {p:.4f}', fontweight='bold')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.2)
plt.suptitle('Manifold Metrics vs Clinical Scores', fontweight='bold')
plt.tight_layout(); plt.subplots_adjust(top=0.86); plt.show()

## 6. Group Comparisons

In [ ]:
df_m['group'] = df_m['condition'].apply(lambda x: 'Healthy' if x=='healthy' else 'Patient')

print(f'{"Metric":<25s} {"Healthy":>10s} {"Patient":>10s} {"p":>10s} {"Effect":>8s}')
print('-' * 65)
for mc in m_cols:
    g_h = df_m[df_m['group']=='Healthy'][mc].dropna()
    g_p = df_m[df_m['group']=='Patient'][mc].dropna()
    stat, pval = mannwhitneyu(g_h, g_p, alternative='two-sided')
    effect = 1 - (2*stat)/(len(g_h)*len(g_p))
    sig = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else ''
    print(f'{mc:<25s} {g_h.mean():>10.2f} {g_p.mean():>10.2f} {pval:>10.4f} {effect:>+8.3f} {sig}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
order = ['healthy','post_rt_mild','post_rt_moderate','post_rt_severe']
for ax, mc in zip(axes, m_cols):
    avail = [c for c in order if c in df_m['condition'].unique()]
    sns.boxplot(data=df_m, x='condition', y=mc, hue='condition', order=avail, palette=cmap, ax=ax, legend=False, width=0.6)
    ax.set_title(mc.replace('_',' ').title(), fontweight='bold', fontsize=10)
    ax.set_xlabel(''); ax.tick_params(axis='x', rotation=25, labelsize=7)
plt.suptitle('Metrics by Radiation Severity', fontweight='bold')
plt.tight_layout(); plt.subplots_adjust(top=0.88); plt.show()